# Train-on × test-on grid: Jackson, Danenberg and both cohorts (11 shared markers)

**Question.** How much of the Jackson → Danenberg drop is a transfer gap, and does training on both cohorts close it?
- **Within-cohort ceiling:** train on Danenberg, test on held-out Danenberg patients.
- **Pooling:** train on Jackson + Danenberg, test on held-out patients of each cohort.

**Grid.** Train on {Jackson, Danenberg, Both} × test on {held-out Jackson patients, held-out Danenberg patients}.

**Models** (one per target marker; same as `model-comparison`):
- Table (LightGBM);
- MLP;
- GCN (Kipf);
- GraphSAGE.

Table + ring is dropped because it tied the Table model.

**Inputs:** the other 10 shared markers + H3, DNA1, DNA2 + log cell area.
- Values are arcsinh(x/5), harmonised per cohort (median, p5–p95).
- The target is the cohort's own per-cell value, harmonised the same way, so the two cohorts can be pooled.
- Jackson values come from the published table; Danenberg values are per-cell image means.
- **Graph:** the 10 nearest cells per cell, within each image, symmetrised.

**Splits:**
- **Jackson:** patients 70 / 10 / 20 (train / validation / test), redrawn per replicate. These are the same splits as `model-comparison` (replicate 0) and `model-comparison-reps` (replicates 1–2).
- **Danenberg:** the frozen test patients from `danenberg-cells-prep`. Train and validation are redrawn per replicate from the remaining patients (7:1).
- **Both:** the union of the two training sets, with early stopping on the pooled validation set.

**Scores:**
- R4 = per-cell r after removing each image's mean; also plain per-cell r.
- 1000× patient bootstrap over test patients; paired differences.
- **Danenberg truth:**
  - the per-cell image mean (`cell mean`, primary; the space the Danenberg models are trained in);
  - the kill-test target map (`target map`, for continuity with earlier runs).
- ERα is reported but left out of medians (different ER antibodies).

**Speed** (same maths as `model-comparison`):
- the neighbour aggregation of the input features is precomputed;
- the sparse products use CSR with a precomputed transpose.

A time guard skips whole (replicate, training set) units that would pass about 11 h, so the kernel still ends COMPLETE.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Reuse `kill-test` per-cell truth and ResNet-UNet predictions; recompute Danenberg per-cell image means

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
RESNET = {(a, m): KT[f"pred__{a}__{m}"].values.astype(np.float32) for a in ARMS for m in MARKERS}
del KT

tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def cell_mean(flat, pm):
    v, loc, n, _ = pm
    cnt = np.bincount(loc, minlength=n)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, np.bincount(loc, flat[v], n) / np.maximum(cnt, 1), np.nan).astype(np.float32)

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    pm = (v, loc[v], len(obj), m.shape)
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    return np.stack([cell_mean(im[:, c], pm) for c in range(im.shape[1])], 1)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(IMGS, ST[:-1], ST[1:])))
assert RAW.shape == (N, len(D_CHAN))
tick("kill-test outputs loaded; per-cell image means recomputed")

## Features and graphs (both cohorts)

In [ ]:
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch
import torch.nn as nn
import torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
K_NN = 10
EPOCHS = 30 if SMOKE else 400
PATIENCE = 10 if SMOKE else 40
HID = 32 if SMOKE else 128
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SOURCES = ["Jackson", "Danenberg", "Both"]
MODELS = ["Table", "MLP", "GCN", "GraphSAGE"]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
XD_in = np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF).astype(np.float32)
XJ_in = np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF).astype(np.float32)
H = {"Jackson": np.c_[harm(XJ_in), harm(DJ["log_area"][:, None])],
     "Danenberg": np.c_[harm(XD_in), harm(DD["log_area"][:, None])]}
nJ, nD = len(H["Jackson"]), len(H["Danenberg"])
print(f"non-finite targets: Jackson {int((~np.isfinite(XJ_in)).sum())}, Danenberg {int((~np.isfinite(XD_in)).sum())}")

def knn_adj(D):
    n = len(D["cells"])
    rows, cols = [], []
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        k = min(K_NN, b - a - 1)
        _, nb = cKDTree(D["xy"][a:b]).query(D["xy"][a:b], k=k + 1)
        rows.append(np.repeat(np.arange(a, b), k))
        cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(n, n)).tocsr()
    return ((A + A.T) > 0).astype(np.float32).tocsr()

def normalise(A):
    n = A.shape[0]
    deg = np.asarray(A.sum(1)).ravel()
    A_mean = (sp.diags(1 / np.maximum(deg, 1)) @ A).tocsr()                # GraphSAGE mean over neighbours
    Ai = A + sp.eye(n, dtype=np.float32, format="csr")
    dh = 1 / np.sqrt(np.asarray(Ai.sum(1)).ravel())
    A_gcn = (sp.diags(dh) @ Ai @ sp.diags(dh)).tocsr()                     # Kipf normalisation with self-loops
    return {"mean": A_mean, "gcn": A_gcn}, float(deg.mean())

ADJ = {}
ADJ["Jackson"], degJ = normalise(knn_adj(DJ))
ADJ["Danenberg"], degD = normalise(knn_adj(DD))
print(f"kNN graphs: mean degree Jackson {degJ:.1f}, Danenberg {degD:.1f}; device {dev}")

def to_csr(M):
    M = M.tocsr().astype(np.float32)
    M.sort_indices()
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int32)), torch.from_numpy(M.indices.astype(np.int32)),
                                   torch.from_numpy(M.data), size=M.shape).to(dev)

class _SpMM(torch.autograd.Function):
    """A @ X with a precomputed CSR transpose for the backward pass."""
    @staticmethod
    def forward(ctx, X, A, AT):
        ctx.AT = AT
        return A @ X
    @staticmethod
    def backward(ctx, G):
        return ctx.AT @ G.contiguous(), None, None

def spmm(Ab, X):
    return _SpMM.apply(X, Ab[0], Ab[1])

def build_graph(src):
    """GPU tensors for a training/prediction graph: features, A (and A^T), row sums, and A @ features."""
    if src == "Both":
        mats = {k: sp.block_diag([ADJ["Jackson"][k], ADJ["Danenberg"][k]], format="csr") for k in ("mean", "gcn")}
        Xn = np.r_[H["Jackson"], H["Danenberg"]]
    else:
        mats, Xn = ADJ[src], H[src]
    Xn = np.nan_to_num(Xn)                                                 # NaN -> 0, the cohort median
    gr = {"x": torch.from_numpy(Xn).to(dev)}
    for k, M in mats.items():
        A = to_csr(M)
        gr[k] = (A, A if k == "gcn" else to_csr(M.T))
        gr["a1_" + k] = torch.from_numpy(np.asarray(M.sum(1)).ravel().astype(np.float32)).to(dev)
        gr["ax_" + k] = torch.from_numpy((M @ Xn).astype(np.float32)).to(dev)
    return gr

GR = {}
def ensure(keys):
    """Keep only the graphs in keys on the GPU (the pooled graph is large)."""
    for k in [k for k in GR if k not in keys]:
        del GR[k]
    if dev == "cuda":
        torch.cuda.empty_cache()
    for k in keys:
        if k not in GR:
            GR[k] = build_graph(k)

ensure({"Jackson", "Danenberg"})
tick("features and graphs")

## Models (identical maths to `model-comparison`; first-layer aggregation precomputed)

In [ ]:
from torch.utils.checkpoint import checkpoint
CKPT = {"on": False}       # gradient checkpointing (memory only; same maths), used for the pooled graph

def run(fn, *a):
    return checkpoint(fn, *a, use_reentrant=False) if (CKPT["on"] and torch.is_grad_enabled()) else fn(*a)

class MLP(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h), nn.ReLU(), nn.Dropout(p),
                              nn.Linear(h, 1))
    def forward(s, x, ax, A, a1):
        return s.net(x).squeeze(-1)

class GCN(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.l1, s.l2, s.out, s.dp = nn.Linear(d, h), nn.Linear(h, h), nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, ax, A, a1):
        # layer 1: A (x W1 + b1) = (A x) W1 + (A 1) b1, as one product
        l1 = lambda ax, a1: s.dp(torch.relu(F.linear(torch.cat([ax, a1[:, None]], 1),
                                                     torch.cat([s.l1.weight, s.l1.bias[:, None]], 1))))
        l2 = lambda h: torch.relu(spmm(A, s.l2(h)))
        return s.out(run(l2, run(l1, ax, a1))).squeeze(-1)

class SAGE(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.s1, s.n1 = nn.Linear(d, h), nn.Linear(d, h, bias=False)
        s.s2, s.n2 = nn.Linear(h, h), nn.Linear(h, h, bias=False)
        s.out, s.dp = nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, ax, A, a1):
        # layer 1: x S1 + A (x N1) = [x, A x] [S1; N1], as one product
        l1 = lambda x, ax: s.dp(torch.relu(F.linear(torch.cat([x, ax], 1), torch.cat([s.s1.weight, s.n1.weight], 1),
                                                    s.s1.bias)))
        l2 = lambda h: torch.relu(s.s2(h) + spmm(A, s.n2(h)))
        return s.out(run(l2, run(l1, x, ax))).squeeze(-1)

NETS = {"MLP": (MLP, "mean"), "GCN": (GCN, "gcn"), "GraphSAGE": (SAGE, "mean")}

def net_inputs(gr, akey, cols):
    return gr["x"][:, cols], gr["ax_" + akey][:, cols], gr[akey], gr["a1_" + akey]

def train_net(cls, inp, y, tr, va, seed):
    torch.manual_seed(seed)
    m = cls(inp[0].shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3, weight_decay=1e-5)
    tr, va = torch.from_numpy(tr).to(dev), torch.from_numpy(va).to(dev)
    best, state, bad, ep = np.inf, None, 0, 0
    for ep in range(EPOCHS):
        m.train()
        opt.zero_grad()
        loss = ((m(*inp)[tr] - y[tr]) ** 2).mean()
        loss.backward()
        opt.step()
        if ep % 5 == 4:
            m.eval()
            with torch.no_grad():
                v = ((m(*inp)[va] - y[va]) ** 2).mean().item()
            if v < best - 1e-5:
                best, bad = v, 0
                state = {k: t.detach().clone() for k, t in m.state_dict().items()}
            else:
                bad += 5
                if bad >= PATIENCE:
                    break
    if state is not None:
        m.load_state_dict(state)
    m.eval()
    return m, ep + 1, best

## Scoring helpers

In [ ]:
def stats_fn(img, pcode, n_pat, W, center):
    def f(a, b):
        ok = np.isfinite(a) & np.isfinite(b)
        a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), img[ok], pcode[ok]
        if center:
            k = int(img.max()) + 1
            cnt = np.maximum(np.bincount(im, minlength=k), 1)
            a = a - (np.bincount(im, a, k) / cnt)[im]
            b = b - (np.bincount(im, b, k) / cnt)[im]
        else:
            a, b = a - a.mean(), b - b.mean()
        sxy, sxx, syy = (np.bincount(pc, a * b, n_pat), np.bincount(pc, a * a, n_pat), np.bincount(pc, b * b, n_pat))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))
    return f

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def scorers(D, te, seed):
    tp, tcode = np.unique(D["patient"][te], return_inverse=True)
    Wt = np.random.default_rng(seed).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    return {"R4": stats_fn(D["img"][te], tcode, len(tp), Wt, True), "r": stats_fn(D["img"][te], tcode, len(tp), Wt, False)}, len(tp)

# Danenberg: frozen test patients (fixed across replicates)
D_TE = np.flatnonzero(DD["split"] == "test")
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
SC_D, n_tpD = scorers(DD, D_TE, SEED + 13)
TRUTH_D = {m: {"cell mean": XD_in[D_TE, IN_SET.index(m)], "target map": TRUTH[m][D_TE]} for m in MARKERS}
print(f"Danenberg test: {len(D_TE):,} cells, {n_tpD} patients")

## Grid: replicates × training set

In [ ]:
EST = {"single": 0.0, "Both": 0.0}
SKIPPED = []
for REP in REPS:
    tick(f"replicate {REP}")
    # Jackson split (same draw as model-comparison / model-comparison-reps)
    jp, jcode = np.unique(DJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    # Danenberg train / validation redrawn among non-test patients
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(DD["split"] == "test", "test",
                       np.where(pd.Series(DD["patient"]).isin(dval).values, "val", "train"))
    IDX = {"Jackson": {s: np.flatnonzero(J_SPLIT == s) for s in ("train", "val", "test")},
           "Danenberg": {s: np.flatnonzero(D_SPLIT == s) for s in ("train", "val", "test")}}
    assert np.array_equal(IDX["Danenberg"]["test"], D_TE)
    for c in IDX:
        pc = DJ["patient"] if c == "Jackson" else DD["patient"]
        print(c, {s: (len(v), len(set(pc[v]))) for s, v in IDX[c].items()}, "(cells, patients)")
    J_TE = IDX["Jackson"]["test"]
    SC_J, n_tpJ = scorers(DJ, J_TE, SEED + 12)
    SC = {("Jackson", "table"): SC_J, ("Danenberg", "cell mean"): SC_D, ("Danenberg", "target map"): SC_D}

    PRED, LOG, ROWS, BOOT = {}, [], [], {}
    for src in SOURCES:
        el = (time.time() - T0) / 60
        proj = EST["Both"] if src == "Both" else EST["single"]
        if el + proj > BUDGET_MIN:
            print(f"!!! skipping replicate {REP}, train on {src}: {el:.0f} min elapsed + {proj:.0f} projected > {BUDGET_MIN:.0f}")
            SKIPPED.append((REP, src))
            continue
        t_src = time.time()
        ensure({"Both"} if src == "Both" else {"Jackson", "Danenberg"})
        CKPT["on"] = src == "Both" or SMOKE
        if src == "Both":
            tr = np.r_[IDX["Jackson"]["train"], IDX["Danenberg"]["train"] + nJ]
            va = np.r_[IDX["Jackson"]["val"], IDX["Danenberg"]["val"] + nJ]
            Hs = np.r_[H["Jackson"], H["Danenberg"]]
        else:
            tr, va, Hs = IDX[src]["train"], IDX[src]["val"], H[src]
        for m in MARKERS:
            k = IN_SET.index(m)
            cols = [i for i in range(Hs.shape[1]) if i != k]               # other 13 channels + log area
            y_all = Hs[:, k]
            trm, vam = tr[np.isfinite(y_all[tr])], va[np.isfinite(y_all[va])]
            mu, sd = y_all[trm].mean(), y_all[trm].std() + 1e-6
            # Table (LightGBM): up to N_FIT_FINAL training cells per cohort
            rs = np.random.default_rng(SEED + REP)
            if src == "Both":
                pools = [trm[trm < nJ], trm[trm >= nJ]]
            else:
                pools = [trm]
            fit = np.sort(np.concatenate([rs.choice(p, min(N_FIT_FINAL, len(p)), replace=False) for p in pools]))
            t0 = time.time()
            mdl = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Hs[fit][:, cols], y_all[fit])
            PRED[(src, "Table", m, "Jackson")] = mdl.predict(H["Jackson"][J_TE][:, cols]).astype(np.float32)
            PRED[(src, "Table", m, "Danenberg")] = mdl.predict(H["Danenberg"][D_TE][:, cols]).astype(np.float32)
            LOG.append(dict(rep=REP, train=src, marker=m, model="Table", epochs=np.nan, best_val_mse_std=np.nan,
                            seconds=time.time() - t0, n_train=len(fit)))
            # neural models
            yt = torch.from_numpy(np.nan_to_num((y_all - mu) / sd)).to(dev)
            for name, (cls, akey) in NETS.items():
                t0 = time.time()
                net, n_ep, best = train_net(cls, net_inputs(GR[src], akey, cols), yt, trm, vam, SEED + REP)
                with torch.no_grad():
                    if src == "Both":
                        p = net(*net_inputs(GR["Both"], akey, cols)).cpu().numpy() * sd + mu
                        pj, pdn = p[:nJ], p[nJ:]
                    else:
                        po = net(*net_inputs(GR[src], akey, cols)).cpu().numpy() * sd + mu
                        pt = net(*net_inputs(GR[OTHER[src]], akey, cols)).cpu().numpy() * sd + mu
                        pj, pdn = (po, pt) if src == "Jackson" else (pt, po)
                PRED[(src, name, m, "Jackson")] = pj[J_TE].astype(np.float32)
                PRED[(src, name, m, "Danenberg")] = pdn[D_TE].astype(np.float32)
                LOG.append(dict(rep=REP, train=src, marker=m, model=name, epochs=n_ep, best_val_mse_std=best,
                                seconds=time.time() - t0, n_train=len(trm)))
                del net
                if dev == "cuda":
                    torch.cuda.empty_cache()
            tick(f"rep {REP}, train on {src}: trained {m}")
        dur = (time.time() - t_src) / 60
        if src == "Both":
            EST["Both"] = dur
        else:
            EST["single"] = max(EST["single"], dur)
            EST["Both"] = max(EST["Both"], 2.2 * EST["single"])
        pd.DataFrame(LOG).to_csv(f"{OUT}/training_log_rep{REP}.csv", index=False)

    # ---- scores (test cells only) ----
    for m in MARKERS:
        truths = {("Jackson", "table"): XJ_in[J_TE, IN_SET.index(m)],
                  ("Danenberg", "cell mean"): TRUTH_D[m]["cell mean"], ("Danenberg", "target map"): TRUTH_D[m]["target map"]}
        cands = [(s, M) for s in SOURCES for M in MODELS if (s, M, m, "Jackson") in PRED]
        for (test, tname), yv in truths.items():
            items = [(s, M, PRED[(s, M, m, test)]) for s, M in cands]
            if test == "Danenberg":
                items += [("Jackson (images)", f"ResNet-UNet {a}", RESNET[(a, m)][D_TE]) for a in ARMS]
            for s, M, p in items:
                for sc, f in SC[(test, tname)].items():
                    pt, bv = f(yv, p)
                    BOOT[(test, tname, s, M, m, sc)] = (pt, bv)
                    lo, hi = ci(bv)
                    ROWS.append(dict(rep=REP, train=s, test=test, truth=tname, marker=m, model=M, score=sc,
                                     value=pt, lo=lo, hi=hi))
    RES = pd.DataFrame(ROWS)
    RES.to_csv(f"{OUT}/scores_rep{REP}.csv", index=False)

    # ---- paired differences (R4) ----
    DIFF = []
    def add(kind, test, tname, m, A, B):
        ka, kb = (test, tname, A[0], A[1], m, "R4"), (test, tname, B[0], B[1], m, "R4")
        if ka not in BOOT or kb not in BOOT:
            return
        lo, hi = ci(BOOT[ka][1] - BOOT[kb][1])
        DIFF.append(dict(rep=REP, kind=kind, test=test, truth=tname, marker=m, train_a=A[0], model_a=A[1],
                         train_b=B[0], model_b=B[1], diff=BOOT[ka][0] - BOOT[kb][0], lo=lo, hi=hi,
                         call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
    for (test, tname) in SC:
        own, oth = test, OTHER[test]
        for m in MARKERS:
            for M in MODELS:
                add("own − other", test, tname, m, (own, M), (oth, M))
                add("both − own", test, tname, m, ("Both", M), (own, M))
                add("both − other", test, tname, m, ("Both", M), (oth, M))
            for s in SOURCES:
                for M in MODELS:
                    for B in ("Table", "MLP"):
                        if M != B:
                            add("model", test, tname, m, (s, M), (s, B))
            if test == "Danenberg":
                for a in ARMS:
                    for s in SOURCES:
                        add("image", test, tname, m, ("Jackson (images)", f"ResNet-UNet {a}"), (s, "Table"))
    DIFF = pd.DataFrame(DIFF)
    DIFF.to_csv(f"{OUT}/differences_R4_rep{REP}.csv", index=False)

    # ---- printout ----
    r4 = RES[(RES.score == "R4") & (RES.marker != "estrogen_receptor_alpha")]
    med = r4.groupby(["test", "truth", "train", "model"]).value.median().unstack("model")
    print(f"--- replicate {REP}: median R4 over 10 markers (ERα excluded)")
    print(med.round(3).to_string())
    if len(DIFF):
        cnt = (DIFF[(DIFF.marker != "estrogen_receptor_alpha") & DIFF.kind.isin(["own − other", "both − own", "both − other"])]
               .groupby(["test", "truth", "kind", "model_a"]).call.value_counts().unstack(fill_value=0))
        print(cnt.to_string())
    tick(f"replicate {REP} done")

print("skipped units:", SKIPPED if SKIPPED else "none")
json.dump({"skipped": SKIPPED, "reps": REPS, "budget_min": BUDGET_MIN, "minutes": (time.time() - T0) / 60},
          open(f"{OUT}/run_info.json", "w"))

## Across replicates: median R4 (mean ± SD)

In [ ]:
fs = sorted(glob.glob(f"{OUT}/scores_rep*.csv"))
if fs:
    ALL = pd.concat([pd.read_csv(f) for f in fs])
    ALL = ALL[(ALL.score == "R4") & (ALL.marker != "estrogen_receptor_alpha")]
    M_ = ALL.groupby(["test", "truth", "train", "model", "rep"]).value.median().reset_index()
    AGG = M_.groupby(["test", "truth", "train", "model"]).value.agg(["mean", "std", "count"]).reset_index()
    AGG.to_csv(f"{OUT}/median_r4_summary.csv", index=False)
    print(AGG.round(3).to_string(index=False))